# 🔢 Multi-Query Generation — Ask the Same Question Many Ways!

## The Core Idea

When you search for something, you have **one** way of phrasing it.
But the document you're looking for might use **completely different words**.

**Multi-query generation solves this by generating multiple versions of the same question.**

```
User Query:  "How do I speed up my ML model?"
                          ↓
              Multi-Query Generator
                          ↓
  Query 1: "How do I speed up my ML model?"
  Query 2: "machine learning model optimization techniques"
  Query 3: "reduce training time neural network"
  Query 4: "faster inference deep learning"
                          ↓
          Search with ALL queries
                          ↓
          Merge & Deduplicate results
                          ↓
          🎯 Better, broader results!
```

## Why Does This Work?

**Embeddings are sensitive to phrasing.** The same idea phrased differently can land in a
slightly different part of the vector space — and therefore retrieve different documents.

By casting a **wider net** with multiple queries, you catch documents that any single query would miss.

## What You'll Learn

1. **The problem** — why one query isn't always enough
2. **Manual variations** — write query variants by hand
3. **Template-based generation** — programmatic variations
4. **LLM-based generation** — let AI write the queries
5. **Merging results** — how to combine results from multiple queries
6. **Reciprocal Rank Fusion (RRF)** — the smart way to merge
7. **Full pipeline** — put it all together
8. **When to use it** — trade-offs and tips

---
**Prerequisites:** You should be familiar with basic embedding-based retrieval.  
If not, check out `../2_retrieval/01_Dense_Retrieval_Embeddings.ipynb` first!

In [37]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()  # reads ANTHROPIC_API_KEY from .env if present

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def llm(prompt, max_tokens=400):
    """Call Claude Haiku. Fast and cheap — perfect for RAG pipelines."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()

print("✅ Anthropic client ready!")
print("   Model: claude-haiku-4-5-20251001")
print("   Set ANTHROPIC_API_KEY in .env or environment.")
print(client)


✅ Anthropic client ready!
   Model: claude-haiku-4-5-20251001
   Set ANTHROPIC_API_KEY in .env or environment.


## Setup — Install and Import

In [38]:
# Install dependencies (run once)
# !pip install sentence-transformers numpy

In [39]:
# Core imports
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ Imports successful!")

✅ Imports successful!


## 1. Build a Knowledge Base

We need a set of documents to search over. Let's create a realistic ML/AI knowledge base.

In [40]:
# Our document collection (knowledge base)
# These are the documents we want to retrieve from
documents = [
    # ML optimization topics
    "Gradient descent is an optimization algorithm that minimizes loss by updating model weights iteratively.",
    "Batch normalization speeds up training and stabilizes deep neural networks.",
    "Learning rate scheduling reduces the learning rate over time to improve convergence.",
    "Mixed precision training uses float16 to reduce memory usage and speed up computation.",
    "Knowledge distillation compresses a large model (teacher) into a smaller model (student).",
    "Quantization reduces model size by using lower-precision numbers (e.g., int8 instead of float32).",
    "Pruning removes unnecessary weights from a neural network to make it smaller and faster.",
    "Early stopping halts training when validation loss stops improving to prevent overfitting.",
    
    # Data topics
    "Data augmentation artificially expands training datasets using transformations like flipping and cropping.",
    "Transfer learning reuses weights from a model trained on one task to accelerate training on another.",
    
    # RAG-specific topics
    "Retrieval-augmented generation (RAG) reduces hallucination by grounding LLM responses in real documents.",
    "Chunking splits long documents into smaller pieces for more precise retrieval in RAG systems.",
    "Embedding models convert text into vectors that capture semantic meaning for similarity search.",
    "BM25 is a keyword-based ranking algorithm widely used in search engines and RAG pipelines.",
    "Hybrid search combines dense (embedding) and sparse (BM25) retrieval for better accuracy.",
]

print(f"Knowledge base has {len(documents)} documents")
print("\nSample documents:")
for i, doc in enumerate(documents[:3]):
    print(f"  [{i}] {doc}")

Knowledge base has 15 documents

Sample documents:
  [0] Gradient descent is an optimization algorithm that minimizes loss by updating model weights iteratively.
  [1] Batch normalization speeds up training and stabilizes deep neural networks.
  [2] Learning rate scheduling reduces the learning rate over time to improve convergence.


In [41]:
# Load the embedding model
# all-MiniLM-L6-v2 is small, fast, and great for learning
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all documents upfront (we do this once, not per query)
doc_embeddings = model.encode(documents)

print(f"Encoded {len(documents)} documents")
print(f"Each embedding has {doc_embeddings.shape[1]} dimensions")

Encoded 15 documents
Each embedding has 384 dimensions


## 2. The Problem — One Query Isn't Always Enough

Let's see where a single query fails to find all relevant documents.

In [42]:
# A simple search function — we'll use this throughout the notebook
def search(query, top_k=5):
    """
    Search the knowledge base with a single query.
    Returns top_k most similar documents with their scores.
    """
    # Step 1: Convert the query to a vector
    query_embedding = model.encode(query)
    
    # Step 2: Compute similarity between query and all documents
    scores = cosine_similarity([query_embedding], doc_embeddings)[0]
    
    # Step 3: Get indices sorted by score (highest first)
    ranked_indices = np.argsort(scores)[::-1][:top_k]
    
    # Step 4: Return documents with their scores and indices
    results = []
    for rank, idx in enumerate(ranked_indices):
        results.append({
            'rank': rank + 1,
            'doc_index': int(idx),
            'score': float(scores[idx]),
            'document': documents[idx]
        })
    return results


# Test with a user query
user_query = "How do I speed up my ML model?"

results = search(user_query, top_k=5)

print(f"Query: '{user_query}'\n")
print("Top 5 Results (Single Query):")
print("=" * 70)
for r in results:
    print(f"Rank {r['rank']} | Score: {r['score']:.4f} | {r['document'][:65]}...")

Query: 'How do I speed up my ML model?'

Top 5 Results (Single Query):
Rank 1 | Score: 0.4232 | Batch normalization speeds up training and stabilizes deep neural...
Rank 2 | Score: 0.3423 | Gradient descent is an optimization algorithm that minimizes loss...
Rank 3 | Score: 0.3305 | Learning rate scheduling reduces the learning rate over time to i...
Rank 4 | Score: 0.3275 | Quantization reduces model size by using lower-precision numbers ...
Rank 5 | Score: 0.3236 | Transfer learning reuses weights from a model trained on one task...


In [43]:
# Now let's see what documents we MISSED
# These are docs that are relevant but didn't show up in top 5

retrieved_indices = {r['doc_index'] for r in results}

# Documents we consider relevant to 'speed up ML model'
# (In real systems you'd have ground truth labels)
actually_relevant_indices = {0, 1, 2, 3, 4, 5, 6}  # The optimization docs

missed = actually_relevant_indices - retrieved_indices

print("Documents we MISSED with a single query:")
print("=" * 70)
for idx in missed:
    print(f"  [Doc {idx}] {documents[idx]}")

print(f"\n⚠️  Missed {len(missed)} relevant documents with just one query!")
print("   These documents use different vocabulary than our query.")

Documents we MISSED with a single query:
  [Doc 3] Mixed precision training uses float16 to reduce memory usage and speed up computation.
  [Doc 4] Knowledge distillation compresses a large model (teacher) into a smaller model (student).
  [Doc 6] Pruning removes unnecessary weights from a neural network to make it smaller and faster.

⚠️  Missed 3 relevant documents with just one query!
   These documents use different vocabulary than our query.


## 3. Manual Query Variations — The Simplest Approach

The most straightforward approach: write several versions of the same query by hand.

In [44]:
# Manually crafted query variations for our question
# Notice: same intent, different vocabulary

manual_variations = [
    "How do I speed up my ML model?",          # Original
    "machine learning model optimization",      # More technical
    "reduce neural network training time",      # Specific to training
    "faster deep learning inference",           # Specific to inference
    "compress model to run faster",             # Compression angle
]

print("Query variations we'll search with:")
for i, q in enumerate(manual_variations, 1):
    print(f"  {i}. {q}")

print(f"\n{len(manual_variations)} queries instead of 1 — 5x wider coverage!")

Query variations we'll search with:
  1. How do I speed up my ML model?
  2. machine learning model optimization
  3. reduce neural network training time
  4. faster deep learning inference
  5. compress model to run faster

5 queries instead of 1 — 5x wider coverage!


In [45]:
# Search with each variation and collect all results

all_results = {}  # doc_index → list of scores from different queries

for query in manual_variations:
    results = search(query, top_k=5)
    
    # Collect scores for each document
    for r in results:
        idx = r['doc_index']
        if idx not in all_results:
            all_results[idx] = []
        all_results[idx].append(r['score'])

# Show what each query found
print("Documents found by each query:\n")
for i, query in enumerate(manual_variations):
    results = search(query, top_k=3)
    top_docs = [r['doc_index'] for r in results]
    print(f"Query {i+1}: '{query[:40]}...'")
    print(f"  → Found docs: {top_docs}")

print(f"\n✅ Combined, we found {len(all_results)} unique documents!")

Documents found by each query:

Query 1: 'How do I speed up my ML model?...'
  → Found docs: [1, 0, 2]
Query 2: 'machine learning model optimization...'
  → Found docs: [0, 2, 9]
Query 3: 'reduce neural network training time...'
  → Found docs: [2, 6, 1]


Query 4: 'faster deep learning inference...'
  → Found docs: [1, 9, 8]
Query 5: 'compress model to run faster...'
  → Found docs: [5, 4, 1]

✅ Combined, we found 9 unique documents!


## 4. How to Merge Results — The Naive Way

Okay, we have results from 5 queries. How do we combine them?

**Naive approach:** Take the maximum score a document got across all queries.

In [46]:
# Naive merge: use the MAX score a document received from any query
# If doc appeared in 3 queries with scores [0.7, 0.6, 0.8], its score = 0.8

merged_scores = {}

for query in manual_variations:
    results = search(query, top_k=5)
    
    for r in results:
        idx = r['doc_index']
        score = r['score']
        
        # Keep the HIGHEST score this document received
        if idx not in merged_scores or score > merged_scores[idx]:
            merged_scores[idx] = score

# Sort by score
ranked = sorted(merged_scores.items(), key=lambda x: x[1], reverse=True)

print("Multi-Query Results (Max Score Merge):")
print("=" * 70)
for rank, (idx, score) in enumerate(ranked[:7], 1):
    print(f"Rank {rank} | Score: {score:.4f} | {documents[idx][:65]}...")

Multi-Query Results (Max Score Merge):
Rank 1 | Score: 0.6082 | Learning rate scheduling reduces the learning rate over time to i...
Rank 2 | Score: 0.5353 | Batch normalization speeds up training and stabilizes deep neural...
Rank 3 | Score: 0.4804 | Gradient descent is an optimization algorithm that minimizes loss...
Rank 4 | Score: 0.4637 | Pruning removes unnecessary weights from a neural network to make...
Rank 5 | Score: 0.4156 | Early stopping halts training when validation loss stops improvin...
Rank 6 | Score: 0.4026 | Transfer learning reuses weights from a model trained on one task...
Rank 7 | Score: 0.3534 | Quantization reduces model size by using lower-precision numbers ...


In [47]:
# Compare: single query vs multi-query

single_results = search("How do I speed up my ML model?", top_k=7)
single_indices = [r['doc_index'] for r in single_results]
multi_indices = [idx for idx, _ in ranked[:7]]

print("Single Query found:")
for idx in single_indices:
    print(f"  [Doc {idx:2d}] {documents[idx][:70]}...")

print("\nMulti-Query found:")
for idx in multi_indices:
    print(f"  [Doc {idx:2d}] {documents[idx][:70]}...")

new_found = set(multi_indices) - set(single_indices)
print(f"\n🎯 Multi-query found {len(new_found)} additional relevant docs!")

Single Query found:
  [Doc  1] Batch normalization speeds up training and stabilizes deep neural netw...
  [Doc  0] Gradient descent is an optimization algorithm that minimizes loss by u...
  [Doc  2] Learning rate scheduling reduces the learning rate over time to improv...
  [Doc  5] Quantization reduces model size by using lower-precision numbers (e.g....
  [Doc  9] Transfer learning reuses weights from a model trained on one task to a...
  [Doc  4] Knowledge distillation compresses a large model (teacher) into a small...
  [Doc  6] Pruning removes unnecessary weights from a neural network to make it s...

Multi-Query found:
  [Doc  2] Learning rate scheduling reduces the learning rate over time to improv...
  [Doc  1] Batch normalization speeds up training and stabilizes deep neural netw...
  [Doc  0] Gradient descent is an optimization algorithm that minimizes loss by u...
  [Doc  6] Pruning removes unnecessary weights from a neural network to make it s...
  [Doc  7] Early stopping

## 5. Reciprocal Rank Fusion (RRF) — The Smart Way to Merge

Max score is okay, but **Reciprocal Rank Fusion (RRF)** is much better.

### The Idea:
- A document that ranks **1st** in multiple query results is more reliable than one that scored highest in just one query.
- RRF **rewards consistency across queries**, not just a single high score.

### The Formula:
```
RRF_score(document) = Σ  1 / (k + rank_in_query_i)
                     queries

k = 60  (a constant that dampens the effect of very high ranks)
```

**Example:**
- Doc A: rank 1 in query 1, rank 3 in query 2 → RRF = 1/61 + 1/63 = 0.0321
- Doc B: rank 1 in query 1 only → RRF = 1/61 = 0.0164
- **Doc A wins** because it appeared consistently across multiple queries!

In [48]:
def reciprocal_rank_fusion(query_results_list, k=60):
    """
    Merge results from multiple queries using Reciprocal Rank Fusion.
    
    Args:
        query_results_list: list of result lists, one per query
                            Each result list is [{doc_index, rank, score, document}, ...]
        k: dampening constant (default 60, standard in literature)
    
    Returns:
        List of (doc_index, rrf_score) sorted by rrf_score descending
    """
    rrf_scores = {}  # doc_index → cumulative RRF score
    
    for query_results in query_results_list:
        for result in query_results:
            idx = result['doc_index']
            rank = result['rank']  # 1-based rank (1 = best)
            
            # RRF formula: add 1/(k + rank) to this document's score
            if idx not in rrf_scores:
                rrf_scores[idx] = 0.0
            
            rrf_scores[idx] += 1.0 / (k + rank)
    
    # Sort by RRF score (highest first)
    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return ranked


# Get results from all query variations
all_query_results = []

for query in manual_variations:
    results = search(query, top_k=10)  # Get top 10 per query for RRF
    all_query_results.append(results)

# Apply RRF
rrf_ranked = reciprocal_rank_fusion(all_query_results)

print("Multi-Query Results (Reciprocal Rank Fusion):")
print("=" * 70)
for rank, (idx, rrf_score) in enumerate(rrf_ranked[:8], 1):
    print(f"Rank {rank} | RRF: {rrf_score:.4f} | {documents[idx][:65]}...")

Multi-Query Results (Reciprocal Rank Fusion):
Rank 1 | RRF: 0.0802 | Batch normalization speeds up training and stabilizes deep neural...
Rank 2 | RRF: 0.0789 | Learning rate scheduling reduces the learning rate over time to i...
Rank 3 | RRF: 0.0782 | Transfer learning reuses weights from a model trained on one task...
Rank 4 | RRF: 0.0775 | Gradient descent is an optimization algorithm that minimizes loss...
Rank 5 | RRF: 0.0770 | Pruning removes unnecessary weights from a neural network to make...
Rank 6 | RRF: 0.0768 | Quantization reduces model size by using lower-precision numbers ...
Rank 7 | RRF: 0.0759 | Knowledge distillation compresses a large model (teacher) into a ...
Rank 8 | RRF: 0.0593 | Early stopping halts training when validation loss stops improvin...


In [49]:
# Let's understand WHY RRF ranks the way it does
# by showing how many queries each top result appeared in

# Track which queries each doc appeared in
doc_appearances = {}  # doc_index → list of query names it appeared in

for query, results in zip(manual_variations, all_query_results):
    for r in results:
        idx = r['doc_index']
        if idx not in doc_appearances:
            doc_appearances[idx] = []
        doc_appearances[idx].append(f"{query[:30]}...")

print("Top RRF results — how many queries found them?\n")
for rank, (idx, rrf_score) in enumerate(rrf_ranked[:5], 1):
    appearances = len(doc_appearances.get(idx, []))
    print(f"Rank {rank} | RRF: {rrf_score:.4f} | Found in {appearances}/5 queries")
    print(f"         Document: {documents[idx][:65]}...")
    print()

Top RRF results — how many queries found them?

Rank 1 | RRF: 0.0802 | Found in 5/5 queries
         Document: Batch normalization speeds up training and stabilizes deep neural...

Rank 2 | RRF: 0.0789 | Found in 5/5 queries
         Document: Learning rate scheduling reduces the learning rate over time to i...

Rank 3 | RRF: 0.0782 | Found in 5/5 queries
         Document: Transfer learning reuses weights from a model trained on one task...

Rank 4 | RRF: 0.0775 | Found in 5/5 queries
         Document: Gradient descent is an optimization algorithm that minimizes loss...

Rank 5 | RRF: 0.0770 | Found in 5/5 queries
         Document: Pruning removes unnecessary weights from a neural network to make...



## 6. Template-Based Query Generation — No LLM Needed!

Instead of writing variations by hand, we can use **templates** to generate them automatically.
This works well without any LLM.

In [50]:
# Template patterns for different question types
# The {query} placeholder gets filled with the user's question

QUERY_TEMPLATES = [
    "{query}",                                    # Original — always include
    "What is {query}?",                           # Definition framing
    "Explain {query} in detail",                  # Explanation framing
    "Best practices for {query}",                 # Best practices framing
    "How does {query} work?",                     # Mechanism framing
    "Common techniques in {query}",               # Techniques framing
    "{query} methods and approaches",             # Methods framing
    "Challenges with {query}",                    # Problem framing (surfaces contrast docs)
]


def generate_queries_from_templates(user_query, templates=QUERY_TEMPLATES):
    """
    Generate query variations by inserting the user query into templates.
    """
    generated = []
    for template in templates:
        # Fill in the template
        expanded = template.replace("{query}", user_query)
        generated.append(expanded)
    return generated


# Test it
user_query = "model quantization"
generated_queries = generate_queries_from_templates(user_query)

print(f"User query: '{user_query}'")
print(f"\nGenerated {len(generated_queries)} query variations:")
for i, q in enumerate(generated_queries, 1):
    print(f"  {i}. {q}")

User query: 'model quantization'

Generated 8 query variations:
  1. model quantization
  2. What is model quantization?
  3. Explain model quantization in detail
  4. Best practices for model quantization
  5. How does model quantization work?
  6. Common techniques in model quantization
  7. model quantization methods and approaches
  8. Challenges with model quantization


In [51]:
# Now search with all generated queries

template_query_results = []
for q in generated_queries:
    results = search(q, top_k=5)
    template_query_results.append(results)

# Merge with RRF
template_rrf = reciprocal_rank_fusion(template_query_results)

print(f"Search: '{user_query}'")
print("\nResults after template-based multi-query:")
print("=" * 70)
for rank, (idx, score) in enumerate(template_rrf[:5], 1):
    print(f"Rank {rank} | RRF: {score:.4f} | {documents[idx]}")

Search: 'model quantization'

Results after template-based multi-query:
Rank 1 | RRF: 0.1311 | Quantization reduces model size by using lower-precision numbers (e.g., int8 instead of float32).
Rank 2 | RRF: 0.1285 | Knowledge distillation compresses a large model (teacher) into a smaller model (student).
Rank 3 | RRF: 0.1262 | Batch normalization speeds up training and stabilizes deep neural networks.
Rank 4 | RRF: 0.1106 | Gradient descent is an optimization algorithm that minimizes loss by updating model weights iteratively.
Rank 5 | RRF: 0.0618 | Data augmentation artificially expands training datasets using transformations like flipping and cropping.


## 7. LLM-Based Query Generation — The Most Powerful Approach

Using an LLM to generate query variations is the most powerful approach because:
- The LLM **understands the intent** behind the query
- It generates **natural, diverse** variations (not just template fills)
- It can reframe the question from **different angles**

### The Prompt Pattern:

```
Generate 4 different versions of this search query.
Each version should have the same intent but different wording.
Output one query per line, nothing else.

Query: {user_query}
```

In [52]:
# Generate multiple query variations using Claude

def generate_query_variations(user_query, num_variations=4):
    """Ask Claude to rephrase the query in different ways."""
    prompt = f"""Generate {num_variations} different ways to ask the same question.
Return ONLY a numbered list, one variation per line.

Original question: {user_query}
Variations:"""
    response = llm(prompt, max_tokens=300)
    variations = [user_query]  # always include original
    for line in response.strip().split('\n'):
        line = line.strip().lstrip('0123456789.). ').strip()
        if line and line != user_query:
            variations.append(line)
    return variations[:num_variations + 1]


# Test it
test_query = "How do I prevent my model from overfitting?"
variations = generate_query_variations(test_query, num_variations=3)

print(f"Original: {test_query}\n")
print("Generated variations:")
for i, v in enumerate(variations, 1):
    print(f"  {i}. {v}")


Original: How do I prevent my model from overfitting?

Generated variations:
  1. How do I prevent my model from overfitting?
  2. What techniques can I use to reduce overfitting in my model?
  3. What are some strategies to keep my model from overfitting?
  4. How can I make sure my model doesn't overfit to the training data?


In [53]:
# Use LLM-generated variations for retrieval

llm_query_results = []
for q in llm_variations:
    results = search(q, top_k=5)
    llm_query_results.append(results)

# Merge with RRF
llm_rrf = reciprocal_rank_fusion(llm_query_results)

print(f"Query: '{user_query}'")
print("\nTop results using LLM-generated multi-query + RRF:")
print("=" * 70)
for rank, (idx, score) in enumerate(llm_rrf[:7], 1):
    print(f"Rank {rank} | RRF: {score:.4f} | {documents[idx][:65]}...")

NameError: name 'llm_variations' is not defined

## 8. Full Pipeline — Putting It All Together

Here's the complete multi-query retrieval pipeline in one clean, readable flow.

In [ ]:
# ============================================================
# COMPLETE MULTI-QUERY RETRIEVAL PIPELINE
# ============================================================

def multi_query_retrieval(user_query, documents, doc_embeddings, model,
                           num_variations=4, top_k_per_query=5, final_top_k=5):
    """
    Full multi-query retrieval pipeline.
    
    Steps:
    1. Generate query variations
    2. Search with each variation
    3. Merge results with Reciprocal Rank Fusion
    4. Return final ranked documents
    """
    
    # ── Step 1: Generate query variations ──
    query_variations = generate_query_variations(user_query, num_variations)
    print(f"Generated {len(query_variations)} query variations")
    
    # ── Step 2: Search with each variation ──
    all_results = []
    
    for variation in query_variations:
        # Embed the query
        q_embedding = model.encode(variation)
        
        # Compute similarity scores
        scores = cosine_similarity([q_embedding], doc_embeddings)[0]
        
        # Get top-k results
        top_indices = np.argsort(scores)[::-1][:top_k_per_query]
        
        results = [
            {'rank': rank + 1, 'doc_index': int(idx),
             'score': float(scores[idx]), 'document': documents[idx]}
            for rank, idx in enumerate(top_indices)
        ]
        all_results.append(results)
    
    # ── Step 3: Merge with RRF ──
    rrf_scores = {}
    k = 60  # RRF constant
    
    for query_results in all_results:
        for result in query_results:
            idx = result['doc_index']
            rank = result['rank']
            rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (k + rank)
    
    # ── Step 4: Sort and return final results ──
    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    final_results = [
        {'rank': i + 1, 'doc_index': idx, 'rrf_score': score, 'document': documents[idx]}
        for i, (idx, score) in enumerate(ranked[:final_top_k])
    ]
    return final_results


# ── Run the full pipeline ──
print("MULTI-QUERY RETRIEVAL PIPELINE")
print("=" * 70)

user_query = "How do I speed up my ML model?"
print(f"User Query: '{user_query}'\n")

final_results = multi_query_retrieval(
    user_query, documents, doc_embeddings, model,
    num_variations=4, top_k_per_query=5, final_top_k=5
)

print("\nFinal Results:")
print("=" * 70)
for r in final_results:
    print(f"Rank {r['rank']} | RRF: {r['rrf_score']:.4f}")
    print(f"         {r['document']}")
    print()

MULTI-QUERY RETRIEVAL PIPELINE
User Query: 'How do I speed up my ML model?'

Generated 5 query variations

Final Results:
Rank 1 | RRF: 0.0804
         Batch normalization speeds up training and stabilizes deep neural networks.

Rank 2 | RRF: 0.0794
         Learning rate scheduling reduces the learning rate over time to improve convergence.

Rank 3 | RRF: 0.0787
         Transfer learning reuses weights from a model trained on one task to accelerate training on another.

Rank 4 | RRF: 0.0635
         Gradient descent is an optimization algorithm that minimizes loss by updating model weights iteratively.

Rank 5 | RRF: 0.0479
         Quantization reduces model size by using lower-precision numbers (e.g., int8 instead of float32).



## 9. Side-by-Side Comparison — Single vs Multi-Query

In [ ]:
# Compare single query vs multi-query side by side

test_queries = [
    "How do I speed up my ML model?",
    "What is RAG?",
    "model quantization",
]

for user_q in test_queries:
    print(f"\n{'='*70}")
    print(f"QUERY: '{user_q}'")
    print(f"{'='*70}")
    
    # Single query
    single = search(user_q, top_k=3)
    single_indices = {r['doc_index'] for r in single}
    
    # Multi-query
    multi = multi_query_retrieval(user_q, documents, doc_embeddings, model,
                                   num_variations=4, top_k_per_query=5, final_top_k=3)
    multi_indices = {r['doc_index'] for r in multi}
    
    print("\n🔍 Single Query top-3:")
    for r in single:
        print(f"   [{r['score']:.3f}] {r['document'][:65]}...")
    
    print("\n🚀 Multi-Query top-3:")
    for r in multi:
        print(f"   [{r['rrf_score']:.4f}] {r['document'][:65]}...")
    
    new_docs = multi_indices - single_indices
    if new_docs:
        print(f"\n✨ Multi-query found {len(new_docs)} additional doc(s)!")
    else:
        print("\n   Same docs retrieved (query was already precise)")


QUERY: 'How do I speed up my ML model?'
Generated 5 query variations

🔍 Single Query top-3:
   [0.423] Batch normalization speeds up training and stabilizes deep neural...
   [0.342] Gradient descent is an optimization algorithm that minimizes loss...
   [0.330] Learning rate scheduling reduces the learning rate over time to i...

🚀 Multi-Query top-3:
   [0.0804] Batch normalization speeds up training and stabilizes deep neural...
   [0.0794] Learning rate scheduling reduces the learning rate over time to i...
   [0.0787] Transfer learning reuses weights from a model trained on one task...

✨ Multi-query found 1 additional doc(s)!

QUERY: 'What is RAG?'
Generated 5 query variations

🔍 Single Query top-3:
   [0.490] Chunking splits long documents into smaller pieces for more preci...
   [0.284] Retrieval-augmented generation (RAG) reduces hallucination by gro...
   [0.242] BM25 is a keyword-based ranking algorithm widely used in search e...

🚀 Multi-Query top-3:
   [0.0645] Retrieval-a

## 10. When to Use Multi-Query (and When Not To)

Multi-query is powerful but not always the right choice.

In [ ]:
# Use this decision guide when designing your RAG pipeline

decision_guide = """
┌─────────────────────────────────────────────────────────────────┐
│            When to Use Multi-Query Generation                   │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ✅ USE IT WHEN:                                                │
│  • Short, ambiguous queries ("speed up model")                  │
│  • Users may use different vocabulary than your docs            │
│  • You need high recall (can't miss relevant docs)              │
│  • Your knowledge base uses technical/formal language           │
│  • Latency is not critical (e.g. batch processing)              │
│                                                                 │
│  ❌ SKIP IT WHEN:                                               │
│  • Query is already very specific / precise                     │
│  • Low latency is critical (real-time chat)                     │
│  • You have very limited API budget (LLM-based generation)      │
│  • Your knowledge base is small (< 100 docs)                    │
│                                                                 │
│  ⚡ LATENCY IMPACT:                                             │
│  • Template-based: +10-30ms (no LLM, just encoding)             │
│  • LLM-based: +200-800ms (depends on model & variations)        │
│  • Can be parallelized! Run queries concurrently.               │
│                                                                 │
│  📈 TYPICAL RECALL IMPROVEMENT: +15% to +35%                    │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
"""

print(decision_guide)


┌─────────────────────────────────────────────────────────────────┐
│            When to Use Multi-Query Generation                   │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ✅ USE IT WHEN:                                                │
│  • Short, ambiguous queries ("speed up model")                  │
│  • Users may use different vocabulary than your docs            │
│  • You need high recall (can't miss relevant docs)              │
│  • Your knowledge base uses technical/formal language           │
│  • Latency is not critical (e.g. batch processing)              │
│                                                                 │
│  ❌ SKIP IT WHEN:                                               │
│  • Query is already very specific / precise                     │
│  • Low latency is critical (real-time chat)                     │
│  • You have very limited API budget (LLM-based 

In [ ]:
# Performance tip: run queries in parallel using threading
# (much faster than sequential when using an LLM for generation)

from concurrent.futures import ThreadPoolExecutor
import time


def search_single(query):
    """Wrapper to make search picklable for threading"""
    q_emb = model.encode(query)
    scores = cosine_similarity([q_emb], doc_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:5]
    return [
        {'rank': rank + 1, 'doc_index': int(idx),
         'score': float(scores[idx]), 'document': documents[idx]}
        for rank, idx in enumerate(top_indices)
    ]


user_query = "How do I speed up my ML model?"
variations = generate_query_variations(user_query, 4)

# Sequential
t0 = time.time()
sequential_results = [search_single(q) for q in variations]
t_seq = time.time() - t0

# Parallel
t0 = time.time()
with ThreadPoolExecutor(max_workers=len(variations)) as executor:
    parallel_results = list(executor.map(search_single, variations))
t_par = time.time() - t0

print(f"Sequential search time: {t_seq*1000:.1f}ms")
print(f"Parallel  search time: {t_par*1000:.1f}ms")
print(f"Speedup: {t_seq/t_par:.1f}x faster with parallel execution!")
print("\n💡 In production, run all query embeddings in one batch:")
print("   embeddings = model.encode(all_queries)  # Much faster than one-by-one!")

Sequential search time: 284.5ms
Parallel  search time: 227.4ms
Speedup: 1.3x faster with parallel execution!

💡 In production, run all query embeddings in one batch:
   embeddings = model.encode(all_queries)  # Much faster than one-by-one!


In [ ]:
# BONUS: Batch encoding all queries at once (most efficient)

user_query = "How do I speed up my ML model?"
variations = generate_query_variations(user_query, 4)

t0 = time.time()

# Encode ALL query variations in one batch call — much faster!
all_query_embeddings = model.encode(variations)  # Shape: (num_queries, dim)

# Compute similarity for all queries at once — also batched!
all_scores = cosine_similarity(all_query_embeddings, doc_embeddings)  # Shape: (num_queries, num_docs)

# Collect results
batch_results = []
for q_idx, scores in enumerate(all_scores):
    top_indices = np.argsort(scores)[::-1][:5]
    results = [
        {'rank': rank + 1, 'doc_index': int(idx),
         'score': float(scores[idx]), 'document': documents[idx]}
        for rank, idx in enumerate(top_indices)
    ]
    batch_results.append(results)

t_batch = time.time() - t0

print(f"Batch encoding time: {t_batch*1000:.1f}ms for {len(variations)} queries")
print("\n✅ All results match:")
print(f"   Single query top doc: {sequential_results[0][0]['document'][:50]}...")
print(f"   Batch  query top doc: {batch_results[0][0]['document'][:50]}...")

Batch encoding time: 333.9ms for 5 queries

✅ All results match:
   Single query top doc: Batch normalization speeds up training and stabili...
   Batch  query top doc: Batch normalization speeds up training and stabili...


## Key Takeaways 🎯

### What You Learned:

| Concept | What It Does |
|---|---|
| **Multi-Query** | Generates multiple phrasings of one query to cast a wider net |
| **Template-based** | Fills query into fixed templates — fast, no LLM needed |
| **LLM-based** | Uses an AI to write natural query variations — most powerful |
| **Max Score Merge** | Simple but ignores consistency across queries |
| **RRF** | Rewards documents that appear consistently across multiple queries |
| **Batch Encoding** | Encode all query variations in one call for max speed |

### The Core Pattern:

```python
# The full multi-query pattern in 5 lines
variations = generate_queries(user_query)            # 1. Generate
embeddings  = model.encode(variations)               # 2. Encode all at once
all_scores  = cosine_similarity(embeddings, doc_emb) # 3. Score all
results     = [get_top_k(s) for s in all_scores]     # 4. Retrieve
final       = reciprocal_rank_fusion(results)        # 5. Merge with RRF
```

### When to Use Which Approach:

```
No LLM budget?    → Template-based generation
Have LLM access?  → LLM-based generation (much better quality)
Merging results?  → Always use RRF over max-score
Worried about speed? → Batch encode + parallel search
```

### Real-World Impact:

- **+15–35% recall improvement** on typical RAG benchmarks
- Used by production systems at **Cohere, LangChain, LlamaIndex**
- Especially powerful for **short or ambiguous queries**

---

## What's Next? 🔜

**`03_HyDE_Hypothetical_Documents.ipynb`** — Instead of expanding the query, generate a *hypothetical answer* and search with that. Often outperforms multi-query for Q&A tasks!

```
Query: "What causes model overfitting?"
   ↓
HyDE generates: "Overfitting occurs when a model learns the training data too
                  well, capturing noise and failing to generalize..."
   ↓
Search with the ANSWER, not the question → better document matches!
```